# Zero-shot Classification (A/B): Plot Builder

Interactive plot builder for the zero-shot binary
classification experiments. Generates matplotlib previews
and copy-pasteable LaTeX (pgfplots) source.

## Experiment structure

Each sweep applies a **single perturbation type** to all
samples. The dropout sweeps set
`active_perturbations=[DROPOUT]` and `noise_std=0`;
the noise sweeps set `active_perturbations=[NOISE]` and
`dropout_rate=0`. So every run has **1000 samples of one
perturbation type**, with 0 baselines.

Each sweep contains **main** and **control** runs:

- **Main** (`dropout_noise`): the option labels are the
  actual perturbation names.
- **Masking/Jitter** (`masking_jitter`): the option labels
  are synonyms of the actual perturbation names.
- **Other controls**: arbitrary labels unrelated to the
  perturbations (`vanilla_chocolate`, `foo_bar`, etc.).

## Metrics

- **accuracy_primary**: fraction of correct picks (first
  token logit). $n = 1000$ per run. SE from Bernoulli.
- **logit_diff_correct**: logit(correct class) minus
  logit(wrong class), averaged over all $n = 1000$ samples
  in the run. SE/SD are empirical (within run). For the
  dropout panel this is dropout minus noise; for the noise
  panel, noise minus dropout.
- **agg_logit_diff_correct**: same using logsumexp
  (aggregate) logits instead of primary.

## Error bands

- **SE**: standard error of the mean.
- **SD**: standard deviation of individual outcomes.

## X axis

Perturbation strengths are shown as **percentiles**
(0 = weakest, 100 = strongest) so that the dropout and
noise panels are comparable despite different raw scales.

In [ ]:
import os
import pathlib

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from wandb_cache import load_sweep

In [ ]:
import yaml

# Sweeps: the zero-shot entries of the manifest sweeps.yaml (project llm-dropout-noise),
# i.e. every zs_<pert> sweep plus its refills zs_<pert>_<suffix>; strengths = the 11 bins of Table 1.
_manifest = yaml.safe_load(
    (_this_dir / "sweeps.yaml").read_text()
)  # cell 1 defines _this_dir
PROJECT = _manifest["project"]


def _zs_ids(pert):
    return [
        sid
        for ids in _manifest.values()
        if isinstance(ids, dict)
        for k, sid in ids.items()
        if k == f"zs_{pert}" or k.startswith(f"zs_{pert}_")
    ]


CLASS_DROPOUT_SWEEPS, CLASS_NOISE_SWEEPS = _zs_ids("dropout"), _zs_ids("noise")

# --- Columns ---
COL_MODEL = "model"
# LaTeX model names: \model{<key>} is defined in the paper's math_commands.tex
MODEL_LATEX = {
    "llama3_8b": r"\model{llama8}",
    "qwen3_14b": r"\model{qwen14}",
    "qwen3_32b": r"\model{qwen32}",
    "olmo3_32b": r"\model{olmo32}",
    "qwen3_8_27b": r"\model{qwen27}",
    "seed_oss_36b": r"\model{seed36}",
    "mistral_small_24b": r"\model{mistral24}",
    "gemma4_31b": r"\model{gemma31}",
}
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"
COL_ALIASES = "aliases"
MAIN_ALIAS = "dropout_noise"

# Aliases drawn with "main" style (solid, thick, markers)
MAIN_STYLE_ALIASES = {"dropout_noise", "masking_jitter"}

# --- Models ---
MODELS = [
    "llama3_8b",
    "qwen3_14b",
    "qwen3_32b",
    "olmo3_32b",
    "qwen3_8_27b",
    "seed_oss_36b",
    "mistral_small_24b",
    "gemma4_31b",
]

# --- Alias colors (from classification_curves_with_control) ---
# Named aliases get fixed colors; others get auto-assigned.
ALIAS_COLORS_MPL = {
    "dropout_noise": "#0000ff",
    "masking_jitter": "#ff0000",
    "rotation_permutation": "#1a5276",
    "scaling_translation": "#2e86c1",
    "vanilla_chocolate": "#1e7a1e",
    "foo_bar": "#7b4a12",
    "x_y": "#4aa3df",
    "none": "#777777",
}
ALIAS_COLORS_LATEX = {  # main aliases: pure blue / pure red; controls as in paper Fig. 4
    "dropout_noise": "blue",
    "masking_jitter": "red",
    "rotation_permutation": "blue!60!black",
    "scaling_translation": "blue!60",
    "vanilla_chocolate": "green!70!black",
    "foo_bar": "brown!70!black",
    "x_y": "cyan!40!blue!70",
    "none": "gray!70",
}
ALIAS_DISPLAY = {
    "dropout_noise": "Dropout/Noise",
    "masking_jitter": "Masking/Jitter",
    "rotation_permutation": "Rotation/Permutation",
    "scaling_translation": "Scaling/Translation",
    "vanilla_chocolate": "Vanilla/Chocolate",
    "foo_bar": "Foo/Bar",
    "x_y": "X/Y",
    "none": "None",
}

DROPOUT_XLABEL = {
    "mpl": "Perturbation percentile (dropout)",
    "latex": "Perturbation percentile (dropout)",
}
NOISE_XLABEL = {
    "mpl": "Perturbation percentile (noise)",
    "latex": "Perturbation percentile (noise)",
}

# --- Metrics ---
# All runs have 1000 samples of a single perturbation type.
# Accuracy: Bernoulli SE with n=1000.
# Logit diff: empirical SE from logged std/sqrt(1000).
METRICS = [
    "accuracy_primary",
    "accuracy_aggregate",
    "accuracy_constrained_primary",
    "accuracy_constrained_aggregate",
    "logit_diff_correct",
    "agg_logit_diff_correct",
    "entropy_mean",  # next-token entropy (nats); per-run SE/SD from entropy_se / entropy_std
]

In [ ]:
# --- Load data ---
def load_and_merge(sweep_ids, project):
    version = 2  # every run must carry metrics_version 2
    dfs = [
        load_sweep(sid, project=project, require_version=version) for sid in sweep_ids
    ]
    return pd.concat(dfs, ignore_index=True)


print("Dropout sweeps:")
df_all_d = load_and_merge(CLASS_DROPOUT_SWEEPS, PROJECT)
print(f"  {len(df_all_d)} runs")

print("Noise sweeps:")
df_all_n = load_and_merge(CLASS_NOISE_SWEEPS, PROJECT)
print(f"  {len(df_all_n)} runs")

# Filter to target models
df_all_d = df_all_d[df_all_d[COL_MODEL].isin(MODELS)].copy()
df_all_n = df_all_n[df_all_n[COL_MODEL].isin(MODELS)].copy()

# --- Determine n per run ---
# Sweep configs set active_perturbations to a single type,
# so all 1000 stochastic samples are the same perturbation.
# num_baselines = 0 (sentences_file is set, no NOTHING class).
if "total_rows" in df_all_d.columns and "num_baselines" in df_all_d.columns:
    N_STOCHASTIC = int(
        (df_all_d["total_rows"] - df_all_d["num_baselines"]).mode().iloc[0]
    )
else:
    N_STOCHASTIC = 1000
print(f"\nSamples per run: {N_STOCHASTIC} (all same perturbation type)")

# --- Show per-model perturbation ranges ---
for name, df, x_col in [
    ("Dropout", df_all_d, COL_DROPOUT),
    ("Noise", df_all_n, COL_NOISE),
]:
    print(f"\n{name} ranges per model:")
    for model in MODELS:
        vals = sorted(df[df[COL_MODEL] == model][x_col].unique())
        if vals:
            print(f"  {model}: {vals[0]:.4f} .. {vals[-1]:.4f} ({len(vals)} points)")

# --- Preprocess: unified logit_diff columns ---
# Dropout sweep: all 1000 samples are DROPOUT.
# The logged "dropout_mean_logit_diff_dropout_vs_noise" is
# mean of (logit_dropout - logit_noise) over all 1000 samples.
# Positive means the model prefers the correct class.
_LD_D = "dropout_mean_logit_diff_dropout_vs_noise"
_LD_D_SE = "dropout_se_logit_diff_dropout_vs_noise"
_LD_D_SD = "dropout_std_logit_diff_dropout_vs_noise"
_ALD_D = "dropout_mean_aggregate_logit_diff_dropout_vs_noise"
_ALD_D_SE = "dropout_se_aggregate_logit_diff_dropout_vs_noise"
_ALD_D_SD = "dropout_std_aggregate_logit_diff_dropout_vs_noise"

df_all_d["logit_diff_correct"] = df_all_d[_LD_D]
df_all_d["logit_diff_correct_se"] = df_all_d[_LD_D_SE]
df_all_d["logit_diff_correct_sd"] = df_all_d[_LD_D_SD]
df_all_d["agg_logit_diff_correct"] = df_all_d[_ALD_D]
df_all_d["agg_logit_diff_correct_se"] = df_all_d[_ALD_D_SE]
df_all_d["agg_logit_diff_correct_sd"] = df_all_d[_ALD_D_SD]

# Noise sweep: all 1000 samples are NOISE.
# The logged column is still (dropout - noise), so we negate
# to get (noise - dropout) = (correct - incorrect).
_LD_N = "noise_mean_logit_diff_dropout_vs_noise"
_LD_N_SE = "noise_se_logit_diff_dropout_vs_noise"
_LD_N_SD = "noise_std_logit_diff_dropout_vs_noise"
_ALD_N = "noise_mean_aggregate_logit_diff_dropout_vs_noise"
_ALD_N_SE = "noise_se_aggregate_logit_diff_dropout_vs_noise"
_ALD_N_SD = "noise_std_aggregate_logit_diff_dropout_vs_noise"

df_all_n["logit_diff_correct"] = -df_all_n[_LD_N]  # negate
df_all_n["logit_diff_correct_se"] = df_all_n[_LD_N_SE]  # SE stays positive
df_all_n["logit_diff_correct_sd"] = df_all_n[_LD_N_SD]
df_all_n["agg_logit_diff_correct"] = -df_all_n[_ALD_N]
df_all_n["agg_logit_diff_correct_se"] = df_all_n[_ALD_N_SE]
df_all_n["agg_logit_diff_correct_sd"] = df_all_n[_ALD_N_SD]

print("\nLogit diff columns added (logit_diff_correct, agg_logit_diff_correct)")

# --- Discover aliases ---
ALL_ALIASES = sorted(df_all_d[COL_ALIASES].dropna().unique())
print(f"\nAliases ({len(ALL_ALIASES)}): {ALL_ALIASES}")

avail_metrics = [m for m in METRICS if m in df_all_d.columns]
print(f"Metrics: {avail_metrics}")

In [ ]:
# ── Helpers ──────────────────────────────────────────────────


def _alias_label(alias):
    return ALIAS_DISPLAY.get(alias, alias)


def _is_main_style(alias):
    return alias in MAIN_STYLE_ALIASES


def _per_model_pct(x_values):
    """Map a sorted array of x values to percentiles 0..100."""
    n = len(x_values)
    if n <= 1:
        return {x_values[0]: 50.0} if n == 1 else {}
    return {v: i * 100 / (n - 1) for i, v in enumerate(x_values)}


# ── Curve computation ────────────────────────────────────────


def compute_curves(df, x_col, metric, selected_aliases, n_stochastic, models):
    """Return {alias: {model: (x_pct, y, se, sd)}}.

    Each run has n_stochastic samples of a single perturbation.
    Percentiles are computed per model (each model has its own
    range of perturbation strengths from localization).

    - accuracy metrics: SE/SD from Bernoulli formula.
    - logit_diff metrics: SE/SD from precomputed columns.
    """
    is_pct = "accuracy" in metric or "f1" in metric
    is_ld = "logit_diff" in metric
    is_entropy = metric == "entropy_mean"
    scale = 100 if is_pct else 1
    se_col, sd_col = (
        ("entropy_se", "entropy_std")
        if is_entropy
        else (f"{metric}_se", f"{metric}_sd")
    )
    result = {}

    for alias in selected_aliases:
        sub_alias = df[df[COL_ALIASES] == alias]
        curves = {}
        for model in models:
            sub = sub_alias[sub_alias[COL_MODEL] == model]
            if metric not in sub.columns or sub.empty:
                continue

            grp = sub.groupby(x_col)
            avg = grp[metric].mean().sort_index() * scale
            if avg.empty:
                continue

            if is_pct:
                p = avg / 100
                pq = p * (1 - p)
                n_runs = grp[metric].count()
                n_total = n_runs * n_stochastic
                se = np.sqrt(pq / n_total) * 100
                sd = np.sqrt(pq) * 100
            elif (is_ld or is_entropy) and se_col in sub.columns:
                n_runs = grp[metric].count()
                if (n_runs > 1).any():
                    se = grp[metric].sem().sort_index()
                    sd = grp[sd_col].mean().sort_index()
                else:
                    se = grp[se_col].first().sort_index()
                    sd = grp[sd_col].first().sort_index()
            else:
                se = grp[metric].sem().sort_index() * scale
                sd = grp[metric].std().sort_index() * scale

            # Per-model percentile: this model's sorted
            # x-values map to 0, 10, 20, ..., 100
            x_raw = avg.index.values
            pct_map = _per_model_pct(x_raw)
            x_pct = np.array([pct_map[v] for v in x_raw])

            curves[model] = (x_pct, avg.values, se.values, sd.values)
        if curves:
            result[alias] = curves
    return result


# ── Matplotlib ───────────────────────────────────────────────


def plot_panel(ax, all_curves, x_label, metric, error_type, ymin, ymax, is_right=False):
    is_pct = "accuracy" in metric or "f1" in metric

    for alias, curves in all_curves.items():
        main_style = _is_main_style(alias)
        c = ALIAS_COLORS_MPL.get(alias, "#888888")
        ls = "-" if main_style else "--"
        lw = 3 if main_style else 1.5
        ms = 4 if main_style else 0
        alpha_band = 0.3 if main_style else 0.2
        zorder_line = 10 if main_style else 1

        for model in MODELS:
            if model not in curves:
                continue
            x, y, se, sd = curves[model]
            err = se if error_type == "SE" else sd
            label = _alias_label(alias) if model == MODELS[0] else None
            ax.plot(
                x,
                y,
                marker="o",
                markersize=ms,
                linewidth=lw,
                color=c,
                label=label,
                linestyle=ls,
                zorder=zorder_line,
            )
            ax.fill_between(
                x, y - err, y + err, alpha=alpha_band, color=c, zorder=zorder_line - 1
            )

    if is_pct:
        ax.axhline(50, color="gray", ls=":", alpha=0.5, label="chance")
    ax.set_xlabel(x_label)
    if is_right:
        ax.set_ylabel("")
        ax.set_yticklabels([])
    else:
        ylabel = (
            "Accuracy (%)"
            if is_pct
            else "Logit diff (correct \u2212 incorrect)"
            if "logit_diff" in metric
            else "Entropy (nats)"
            if metric == "entropy_mean"
            else metric
        )
        ax.set_ylabel(ylabel)
    ax.set_xlim(-2, 102)
    ax.set_xticks(range(0, 101, 10))
    ax.grid(True, alpha=0.3)
    if ymin is not None and ymax is not None and ymin < ymax:
        ax.set_ylim(ymin, ymax)


# ── LaTeX / pgfplots ────────────────────────────────────────


def latex_panel(
    all_curves,
    x_label,
    metric,
    error_type,
    models,
    is_right=False,
    ymin=None,
    ymax=None,
):
    is_pct = "accuracy" in metric or "f1" in metric
    is_ld = "logit_diff" in metric

    L = []
    L.append(r"\begin{tikzpicture}")
    L.append(r"\begin{axis}[")
    L.append("    height=7cm,")
    L.append("    width=\\linewidth,")
    L.append("    grid=major,")
    L.append(f"    xlabel={{{x_label}}},")
    if is_right:
        L.append("    ylabel={},")
        L.append("    yticklabels={},")
    else:
        if is_pct:
            L.append("    ylabel={Accuracy (\\%)},")
        elif is_ld:
            L.append("    ylabel={Logit diff (correct $-$ incorrect)},")
        elif metric == "entropy_mean":
            L.append("    ylabel={Entropy (nats)},")
        else:
            L.append(f"    ylabel={{{metric}}},")
    L.append("    xmin=0, xmax=100,")
    L.append("    xtick={0,10,...,100},")
    if ymin is not None and ymax is not None:
        L.append(f"    ymin={ymin}, ymax={ymax},")
    L.append("    legend entries={},")
    L.append("]")

    for alias, curves in all_curves.items():
        main_style = _is_main_style(alias)
        c = ALIAS_COLORS_LATEX.get(alias, "gray")
        # paper style: main aliases solid with dots, controls dashed without marks
        style = (
            "mark=*, mark size=1, line width=1.5pt"
            if main_style
            else "dashed, line width=0.8pt"
        )
        opacity = "0.3" if main_style else "0.15"

        for model in models:
            if model not in curves:
                continue
            x, y, se, sd = curves[model]
            err = se if error_type == "SE" else sd

            up = " ".join(f"({xi:.1f},{yi + ei:.4f})" for xi, yi, ei in zip(x, y, err))
            lo = " ".join(
                f"({xi:.1f},{yi - ei:.4f})"
                for xi, yi, ei in reversed(list(zip(x, y, err)))
            )
            L.append(
                f"\\addplot[{c}, fill={c}, "
                f"fill opacity={opacity}, "
                f"draw=none, forget plot] "
                f"coordinates {{{up} {lo}}} --cycle;"
            )
            coords = " ".join(f"({xi:.1f},{yi:.4f})" for xi, yi in zip(x, y))
            L.append(f"\\addplot[{c}, {style}, forget plot] coordinates {{{coords}}};")

    if is_pct:
        L.append(
            r"\addplot[gray, dashed, line width=0.5pt, "
            r"forget plot] coordinates {(0,50) (100,50)};"
        )
    L.append(r"\end{axis}")
    L.append(r"\end{tikzpicture}")
    return "\n".join(L)


def latex_figure(
    all_curves_d,
    all_curves_n,
    metric,
    error_type,
    models,
    selected_aliases,
    caption,
    label,
    ymin=None,
    ymax=None,
):
    d_tex = latex_panel(
        all_curves_d,
        DROPOUT_XLABEL["latex"],
        metric,
        error_type,
        models,
        ymin=ymin,
        ymax=ymax,
    )
    n_tex = latex_panel(
        all_curves_n,
        NOISE_XLABEL["latex"],
        metric,
        error_type,
        models,
        is_right=True,
        ymin=ymin,
        ymax=ymax,
    )

    items = []
    for a in selected_aliases:
        if a in all_curves_d or a in all_curves_n:
            c = ALIAS_COLORS_LATEX.get(a, "gray")
            al = _alias_label(a).replace("_", r"\_")
            ls = "thick, mark=*, mark size=1.5" if _is_main_style(a) else "dashed"
            items.append(
                f"\\tikz\\draw[{c}, {ls}] plot coordinates "
                f"{{(0,0) (0.4,0)}}; {{\\small {al}}}"
            )
    legend = "\\hspace{1em}".join(items)
    err_tag = "SE" if error_type == "SE" else "SD"

    return (
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"{legend}\n"
        "\\\\[6pt]\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {d_tex}\n"
        f"\\end{{minipage}}\n"
        "\\hfill\n"
        f"\\begin{{minipage}}{{0.48\\textwidth}}\n"
        f"  {n_tex}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{{caption} "
        f"Bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:{label}}}\n"
        "\\end{figure}"
    )

In [ ]:
# ── Interactive plot ──────────────────────────────────────────


# Auto-assign colors for aliases not in ALIAS_COLORS_MPL


_EXTRA_PALETTE = plt.cm.tab20.colors


_color_idx = 0


for a in ALL_ALIASES:
    if a not in ALIAS_COLORS_MPL:
        ALIAS_COLORS_MPL[a] = "#{:02x}{:02x}{:02x}".format(
            *[int(c * 255) for c in _EXTRA_PALETTE[_color_idx % 20][:3]]
        )

        _color_idx += 1


w_metric = widgets.Dropdown(
    options=avail_metrics, value=avail_metrics[0], description="Metric:"
)


w_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)


w_models = {
    m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
}


# Checkboxes for aliases (main always on, controls individually togglable)


w_alias_checks = {}


for a in ALL_ALIASES:
    w_alias_checks[a] = widgets.Checkbox(
        value=(a == MAIN_ALIAS),
        description=_alias_label(a),
        indent=False,
        layout=widgets.Layout(width="180px"),
    )


w_alias_btn_all = widgets.Button(
    description="Select all", layout=widgets.Layout(width="100px")
)


w_alias_btn_none = widgets.Button(
    description="Clear all", layout=widgets.Layout(width="100px")
)


w_alias_btn_main = widgets.Button(
    description="Main only", layout=widgets.Layout(width="100px")
)


def _on_alias_all(_):

    for cb in w_alias_checks.values():
        cb.value = True


def _on_alias_none(_):

    for cb in w_alias_checks.values():
        cb.value = False


def _on_alias_main(_):

    for cb in w_alias_checks.values():
        cb.value = False
    w_alias_checks[MAIN_ALIAS].value = True


w_alias_btn_all.on_click(_on_alias_all)
w_alias_btn_none.on_click(_on_alias_none)
w_alias_btn_main.on_click(_on_alias_main)

w_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
w_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out = widgets.Output()


def redraw(*_):
    out.clear_output(wait=True)
    with out:
        metric = w_metric.value
        et = w_error.value
        models = [m for m in MODELS if w_models[m].value]
        sel = [a for a in ALL_ALIASES if w_alias_checks[a].value]
        ymin = w_ymin.value if not np.isnan(w_ymin.value) else None
        ymax = w_ymax.value if not np.isnan(w_ymax.value) else None

        if not sel:
            print("No aliases selected.")
            return

        print(f"Selected: {sel}")

        cd = compute_curves(df_all_d, COL_DROPOUT, metric, sel, N_STOCHASTIC, models)
        cn = compute_curves(df_all_n, COL_NOISE, metric, sel, N_STOCHASTIC, models)

        if not cd and not cn:
            print("WARNING: no curves produced.")
            return

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
        plot_panel(ax1, cd, DROPOUT_XLABEL["mpl"], metric, et, ymin, ymax)
        plot_panel(ax2, cn, NOISE_XLABEL["mpl"], metric, et, ymin, ymax, is_right=True)

        handles, labels = ax1.get_legend_handles_labels()
        fig.legend(
            handles=handles,
            loc="upper center",
            ncol=min(len(sel), 6),
            fontsize=9,
            bbox_to_anchor=(0.5, 1.08),
        )

        n_sel = len(sel)
        alias_summary = (
            "main only"
            if sel == [MAIN_ALIAS]
            else f"{n_sel} aliases"
            if n_sel > 2
            else " + ".join(_alias_label(a) for a in sel)
        )
        fig.suptitle(
            f"Zero-shot Classification ({alias_summary}, \u00b1{et})",
            fontsize=14,
            y=1.12,
        )
        fig.tight_layout()
        plt.show()

        caption = f"Zero-shot classification ({alias_summary})."
        label_tag = f"class-{n_sel}aliases"
        print(
            latex_figure(
                cd,
                cn,
                metric,
                et,
                models,
                sel,
                caption,
                label_tag,
                ymin=ymin,
                ymax=ymax,
            )
        )


for w in [w_metric, w_error, w_ymin, w_ymax]:
    w.observe(redraw, names="value")
for cb in w_models.values():
    cb.observe(redraw, names="value")
for cb in w_alias_checks.values():
    cb.observe(redraw, names="value")

display(
    widgets.HBox([w_metric, w_error, w_ymin, w_ymax]),
    widgets.HBox(list(w_models.values())),
    widgets.HBox([w_alias_btn_all, w_alias_btn_none, w_alias_btn_main]),
    widgets.GridBox(
        list(w_alias_checks.values()),
        layout=widgets.Layout(grid_template_columns="repeat(5, 180px)"),
    ),
    out,
)
redraw()

## Combined dropout/noise accuracy per model

Same data as above, one alias pair at a time. **Left:** for each model and each percentile $k$ of its
perturbation range, the mean of the dropout-sweep accuracy at the $k$-th dropout strength and the
noise-sweep accuracy at the $k$-th noise strength, i.e. $\tfrac12\,[\mathrm{acc}_{\text{dropout}}(k) + \mathrm{acc}_{\text{noise}}(k)]$.
**Right:** one bar per model, the plain mean of all 22 accuracies (11 dropout + 11 noise), sorted.
Error bars propagate the Bernoulli SE (or SD) of the runs: half the root sum of squares for a point,
and the same over the points for a bar. Duplicate runs at a grid point are averaged first.


In [ ]:
# ── Combined dropout/noise view: per-model curves + per-model bars ─────────────

MODEL_COLORS_MPL = {
    "llama3_8b": "#f4b6c2",
    "qwen3_14b": "#8fae82",
    "qwen3_32b": "#1b3a6b",
    "olmo3_32b": "#8b1a1a",
    "qwen3_8_27b": "#00a3a3",
    "seed_oss_36b": "#b5651d",
    "mistral_small_24b": "#d4a017",
    "gemma4_31b": "#4b86b4",
}
MODEL_COLORS_LATEX = {
    "llama3_8b": "red!40",
    "qwen3_14b": "green!40!black!30",
    "qwen3_32b": "blue!70!black",
    "olmo3_32b": "red!70!black",
    "qwen3_8_27b": "teal!70!black",
    "seed_oss_36b": "brown!80!black",
    "mistral_small_24b": "yellow!60!black",
    "gemma4_31b": "cyan!60!black",
}


def combined_curves(alias, metric, models):
    """{model: (x_pct, y_avg, se_avg, sd_avg)}: mean of the dropout and noise curves at equal percentiles."""
    cd = compute_curves(
        df_all_d, COL_DROPOUT, metric, [alias], N_STOCHASTIC, models
    ).get(alias, {})
    cn = compute_curves(df_all_n, COL_NOISE, metric, [alias], N_STOCHASTIC, models).get(
        alias, {}
    )
    out = {}
    for model in models:
        if model not in cd or model not in cn:
            continue
        d = {round(x, 6): v for x, *v in zip(*cd[model])}
        n = {round(x, 6): v for x, *v in zip(*cn[model])}
        xs = sorted(set(d) & set(n))
        if not xs:
            continue
        y = np.array([(d[x][0] + n[x][0]) / 2 for x in xs])
        se = np.array([np.hypot(d[x][1], n[x][1]) / 2 for x in xs])
        sd = np.array([np.hypot(d[x][2], n[x][2]) / 2 for x in xs])
        out[model] = (np.array(xs), y, se, sd)
    return out


def combined_bars(curves, error_type):
    """[(model, mean over all points, propagated error)] sorted by the mean, descending."""
    rows = []
    for model, (x, y, se, sd) in curves.items():
        err = se if error_type == "SE" else sd
        rows.append((model, float(y.mean()), float(np.sqrt((err**2).sum()) / len(err))))
    return sorted(rows, key=lambda r: -r[1])


def latex_combined(
    curves, bars, alias, metric, error_type, caption, label, ymin=None, ymax=None
):
    ylab = "Accuracy (\\%)" if "accuracy" in metric else metric
    yr = (
        f"    ymin={ymin}, ymax={ymax},\n"
        if ymin is not None and ymax is not None
        else ""
    )
    L = [
        r"\begin{tikzpicture}",
        r"\begin{axis}[",
        "    height=7cm,",
        "    width=\\linewidth,",
        "    grid=major,",
        "    xlabel={Perturbation percentile (dropout and noise)},",
        f"    ylabel={{{ylab}}},",
        "    xmin=0, xmax=100,",
        "    xtick={0,10,...,100},",
    ]
    if yr:
        L.append(yr.rstrip("\n"))
    L.append("]")
    for model, (x, y, se, sd) in curves.items():
        c = MODEL_COLORS_LATEX.get(model, "gray")
        err = se if error_type == "SE" else sd
        up = " ".join(f"({xi:.1f},{yi + ei:.4f})" for xi, yi, ei in zip(x, y, err))
        lo = " ".join(
            f"({xi:.1f},{yi - ei:.4f})" for xi, yi, ei in reversed(list(zip(x, y, err)))
        )
        L.append(
            f"\\addplot[{c}, fill={c}, fill opacity=0.2, draw=none, forget plot] coordinates {{{up} {lo}}} --cycle;"
        )
        coords = " ".join(f"({xi:.1f},{yi:.4f})" for xi, yi in zip(x, y))
        L.append(
            f"\\addplot[{c}, mark=o, mark size=1, line width=1.2pt, forget plot] coordinates {{{coords}}};"
        )
    L.append(
        r"\addplot[gray, dashed, line width=0.5pt, forget plot] coordinates {(0,50) (100,50)};"
    )
    L += [r"\end{axis}", r"\end{tikzpicture}"]
    curve_tex = "\n".join(L)

    names = [m for m, _, _ in bars]
    sym = ",".join(MODEL_LATEX[n] for n in names)
    B = [
        r"\begin{tikzpicture}",
        r"\begin{axis}[",
        "    height=7cm,",
        "    width=\\linewidth,",
        "    ybar,",
        "    bar width=10pt,",
        "    grid=major,",
        f"    ylabel={{Mean {ylab.lower()}}},",
        f"    symbolic x coords={{{sym}}},",
        "    xtick=data,",
        "    x tick label style={rotate=45, anchor=east, font=\\scriptsize},",
        "    enlarge x limits=0.1,",
        "    error bars/.cd, y dir=both, y explicit,",
        "]",
    ]
    for model, mean, err in bars:
        c = MODEL_COLORS_LATEX.get(model, "gray")
        B.append(
            f"\\addplot[fill={c}, draw={c}, forget plot] coordinates {{({MODEL_LATEX[model]},{mean:.4f}) +- (0,{err:.4f})}};"
        )
    B.append(
        r"\addplot[gray, dashed, line width=0.5pt, forget plot, sharp plot] coordinates {("
        + MODEL_LATEX[names[0]]
        + ",50) ("
        + MODEL_LATEX[names[-1]]
        + ",50)};"
    )
    B += [r"\end{axis}", r"\end{tikzpicture}"]
    bar_tex = "\n".join(B)

    legend = "\\hspace{1em}".join(
        f"\\tikz\\draw[{MODEL_COLORS_LATEX.get(m, 'gray')}, thick, mark=o, mark size=1.5] plot coordinates {{(0,0) (0.4,0)}}; {MODEL_LATEX[m]}"
        for m in curves
    )
    err_tag = "SE" if error_type == "SE" else "SD"
    return (
        "\\begin{figure}[ht]\n\\centering\n" + legend + "\n\\\\[6pt]\n"
        "\\begin{minipage}{0.48\\textwidth}\n  "
        + curve_tex
        + "\n\\end{minipage}\n\\hfill\n"
        "\\begin{minipage}{0.48\\textwidth}\n  " + bar_tex + "\n\\end{minipage}\n"
        f"\\caption{{{caption} Labels: {_alias_label(alias).replace('_', chr(92) + '_')}. Bars/bands show $\\pm${err_tag}.}}\n"
        f"\\label{{fig:{label}}}\n\\end{{figure}}"
    )


_acc_metrics = [m for m in avail_metrics if "accuracy" in m]
wc_alias = widgets.Dropdown(
    options=[(_alias_label(a), a) for a in ALL_ALIASES],
    value=MAIN_ALIAS if MAIN_ALIAS in ALL_ALIASES else ALL_ALIASES[0],
    description="Labels:",
)
wc_metric = widgets.Dropdown(
    options=_acc_metrics, value=_acc_metrics[0], description="Metric:"
)
wc_error = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
wc_models = {
    m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
}
wc_ymin = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
wc_ymax = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out_c = widgets.Output()


def redraw_combined(*_):
    out_c.clear_output(wait=True)
    with out_c:
        alias, metric, et = wc_alias.value, wc_metric.value, wc_error.value
        models = [m for m in MODELS if wc_models[m].value]
        ymin = wc_ymin.value if not np.isnan(wc_ymin.value) else None
        ymax = wc_ymax.value if not np.isnan(wc_ymax.value) else None
        curves = combined_curves(alias, metric, models)
        if not curves:
            print("No model has both a dropout and a noise curve for this alias.")
            return
        bars = combined_bars(curves, et)

        fig, (ax1, ax2) = plt.subplots(
            1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [3, 2]}
        )
        for model, (x, y, se, sd) in curves.items():
            err = se if et == "SE" else sd
            c = MODEL_COLORS_MPL.get(model, "#888888")
            ax1.plot(x, y, marker="o", markersize=4, linewidth=2, color=c, label=model)
            ax1.fill_between(x, y - err, y + err, alpha=0.2, color=c)
        ax1.axhline(50, color="gray", ls=":", alpha=0.6)
        ax1.set_xlabel("Perturbation percentile (dropout and noise)")
        ax1.set_ylabel("Accuracy (%)")
        ax1.set_xlim(-2, 102)
        ax1.set_xticks(range(0, 101, 10))
        ax1.grid(True, alpha=0.3)
        ax1.legend(fontsize=9)
        names = [b[0] for b in bars]
        ax2.bar(
            names,
            [b[1] for b in bars],
            yerr=[b[2] for b in bars],
            capsize=3,
            color=[MODEL_COLORS_MPL.get(m, "#888888") for m in names],
        )
        ax2.axhline(50, color="gray", ls=":", alpha=0.6)
        ax2.set_ylabel("Mean accuracy over the 22 points (%)")
        ax2.tick_params(axis="x", rotation=45)
        for lbl in ax2.get_xticklabels():
            lbl.set_ha("right")
        ax2.grid(True, axis="y", alpha=0.3)
        if ymin is not None and ymax is not None and ymin < ymax:
            ax1.set_ylim(ymin, ymax)
            ax2.set_ylim(ymin, ymax)
        fig.suptitle(
            f"Zero-shot, dropout and noise averaged ({_alias_label(alias)}, \u00b1{et})",
            fontsize=14,
        )
        fig.tight_layout()
        plt.show()

        print(
            pd.DataFrame(
                bars, columns=["model", f"mean {metric} (%)", f"{et} (%)"]
            ).to_string(index=False)
        )
        caption = "Zero-shot classification, dropout and noise accuracies averaged at equal percentiles (left) and over the whole range (right)."
        print(
            latex_combined(
                curves,
                bars,
                alias,
                metric,
                et,
                caption,
                f"class-combined-{alias.replace('_', '-')}",
                ymin=ymin,
                ymax=ymax,
            )
        )


for w in [wc_alias, wc_metric, wc_error, wc_ymin, wc_ymax]:
    w.observe(redraw_combined, names="value")
for cb in wc_models.values():
    cb.observe(redraw_combined, names="value")
display(
    widgets.HBox([wc_alias, wc_metric, wc_error, wc_ymin, wc_ymax]),
    widgets.HBox(list(wc_models.values())),
    out_c,
)
redraw_combined()

## Averaged plot

One panel: at each percentile, the mean of the dropout curve (left plot above) and the noise curve
(right plot above). Error bands combine the two SEs (or SDs) as $\sqrt{a^2+b^2}/2$.

In [ ]:
# ── Averaged plot: mean of the dropout and noise curves at each percentile ──


def average_curves(curves_d, curves_n):
    """{alias: {model: (x, y, se, sd)}} with y = (y_dropout + y_noise) / 2 at each x.

    Both curves are on the same percentile grid (0, 10, ..., 100 of each model's range).
    SE/SD of the mean of two independent estimates: sqrt(a^2 + b^2) / 2.
    """
    out = {}
    for alias in curves_d.keys() & curves_n.keys():
        per_model = {}
        for model in curves_d[alias].keys() & curves_n[alias].keys():
            xd, yd, sed, sdd = curves_d[alias][model]
            xn, yn, sen, sdn = curves_n[alias][model]
            if not np.array_equal(xd, xn):
                raise ValueError(
                    f"{alias}/{model}: dropout and noise grids differ: {xd} vs {xn}"
                )
            per_model[model] = (
                xd,
                (yd + yn) / 2,
                np.sqrt(sed**2 + sen**2) / 2,
                np.sqrt(sdd**2 + sdn**2) / 2,
            )
        if per_model:
            out[alias] = per_model
    return out


def latex_figure_single(
    all_curves,
    x_label,
    metric,
    error_type,
    models,
    selected_aliases,
    caption,
    label,
    ymin=None,
    ymax=None,
):
    """One-panel version of latex_figure (same legend and panel emitters)."""
    panel = latex_panel(
        all_curves, x_label, metric, error_type, models, ymin=ymin, ymax=ymax
    )
    items = []
    for a in selected_aliases:
        if a in all_curves:
            c = ALIAS_COLORS_LATEX.get(a, "gray")
            al = _alias_label(a).replace("_", r"\_")
            ls = (
                "thick, mark=*, mark size=1.5"
                if _is_main_style(a)
                else "dashed, line width=0.8pt"
            )
            items.append(
                f"\\mbox{{\\tikz[baseline=-0.6ex]\\draw[{c}, {ls}] plot coordinates {{(0,0) (0.4,0)}}; {{\\small {al}}}}}"
            )
    legend = "\\hspace{1em}%\n".join(items)
    return (
        "\\begin{figure}[ht]\n\\centering\n"
        f"{legend}\\\\[3pt]\n"
        "\\begin{minipage}{0.6\\textwidth}\n"
        f"  {panel}\n"
        "\\end{minipage}\n"
        "\\vspace{-8pt}\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{fig:{label}}}\n"
        "\\end{figure}"
    )


AVG_XLABEL = {
    "mpl": "Perturbation strength (percentile of range, %)",
    "latex": "Perturbation strength (\\%)",
}

w_metric_avg = widgets.Dropdown(
    options=avail_metrics, value=avail_metrics[0], description="Metric:"
)
w_error_avg = widgets.RadioButtons(
    options=["SE", "SD"],
    value="SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
w_models_avg = {
    m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
}
w_alias_checks_avg = {
    a: widgets.Checkbox(
        value=(a == MAIN_ALIAS),
        description=_alias_label(a),
        indent=False,
        layout=widgets.Layout(width="180px"),
    )
    for a in ALL_ALIASES
}
w_ymin_avg = widgets.FloatText(
    value=float("nan"), description="y min:", layout=widgets.Layout(width="150px")
)
w_ymax_avg = widgets.FloatText(
    value=float("nan"), description="y max:", layout=widgets.Layout(width="150px")
)
out_avg = widgets.Output()


def redraw_avg(*_):
    out_avg.clear_output(wait=True)
    with out_avg:
        metric = w_metric_avg.value
        et = w_error_avg.value
        models = [m for m in MODELS if w_models_avg[m].value]
        sel = [a for a in ALL_ALIASES if w_alias_checks_avg[a].value]
        ymin = w_ymin_avg.value if not np.isnan(w_ymin_avg.value) else None
        ymax = w_ymax_avg.value if not np.isnan(w_ymax_avg.value) else None
        if not sel:
            print("No aliases selected.")
            return
        cd = compute_curves(df_all_d, COL_DROPOUT, metric, sel, N_STOCHASTIC, models)
        cn = compute_curves(df_all_n, COL_NOISE, metric, sel, N_STOCHASTIC, models)
        ca = average_curves(cd, cn)
        if not ca:
            print("WARNING: no curves produced.")
            return

        fig, ax = plt.subplots(figsize=(9, 6))
        plot_panel(ax, ca, AVG_XLABEL["mpl"], metric, et, ymin, ymax)
        handles, labels = ax.get_legend_handles_labels()
        fig.legend(
            handles=handles,
            loc="upper center",
            ncol=min(len(sel), 6),
            fontsize=9,
            bbox_to_anchor=(0.5, 1.06),
        )
        fig.suptitle(
            f"Zero-shot classification, dropout and noise averaged (\u00b1{et})",
            fontsize=14,
            y=1.1,
        )
        fig.tight_layout()
        plt.show()

        n_sel = len(sel)
        caption = "Zero-shot classification, average of the dropout and noise curves at each percentile of the perturbation range."
        print(
            latex_figure_single(
                ca,
                AVG_XLABEL["latex"],
                metric,
                et,
                models,
                sel,
                caption,
                f"class-avg-{n_sel}aliases",
                ymin=ymin,
                ymax=ymax,
            )
        )


for w in [w_metric_avg, w_error_avg, w_ymin_avg, w_ymax_avg]:
    w.observe(redraw_avg, names="value")
for cb in list(w_models_avg.values()) + list(w_alias_checks_avg.values()):
    cb.observe(redraw_avg, names="value")

display(
    widgets.HBox([w_metric_avg, w_error_avg, w_ymin_avg, w_ymax_avg]),
    widgets.HBox(list(w_models_avg.values())),
    widgets.GridBox(
        list(w_alias_checks_avg.values()),
        layout=widgets.Layout(grid_template_columns="repeat(5, 180px)"),
    ),
    out_avg,
)
redraw_avg()